# 量化交易系统示例

这个notebook演示如何使用量化交易系统进行回测和分析。

In [ ]:
import sys
sys.path.append('../')

from datetime import datetime
from src.data.tushare_provider import TushareDataProvider
from src.strategies.moving_average_crossover import MovingAverageCrossover
from src.backtesting.engine import BacktestEngine
from src.risk.manager import RiskConfig, RiskManager

import os
from dotenv import load_dotenv
load_dotenv('../.env')

In [ ]:
# 初始化数据源
api_token = os.getenv('TUSHARE_TOKEN')
data_provider = TushareDataProvider(api_key=api_token)

In [ ]:
# 创建策略
strategy = MovingAverageCrossover({
    'fast_period': 5,
    'slow_period': 20
})
strategy

In [ ]:
# 配置风险
risk_config = RiskConfig(
    max_single_position=0.1,  # 单票最大10%
    max_total_drawdown=0.2    # 最大回撤20%
)

In [ ]:
# 创建回测引擎
engine = BacktestEngine(
    data_provider=data_provider,
    strategy=strategy,
    initial_cash=1000000,
    commission_rate=0.0003,
    slippage_pct=0.001,
    risk_config=risk_config
)

In [ ]:
# 运行回测
start = datetime(2020, 1, 1)
end = datetime(2024, 1, 1)
symbols = ['000001.SH', '600000.SH']

results = engine.run(symbols, start, end)
engine.print_results()

In [ ]:
# 绘制净值曲线
from src.analytics.performance import PerformanceAnalyzer
analyzer = PerformanceAnalyzer(results['portfolio'])
analyzer.plot_equity_curve()

In [ ]:
# 查看交易记录
trades_df = analyzer.get_trades_df()
trades_df.head(10)

## 尝试其他策略

试试RSI策略：

In [ ]:
from src.strategies.rsi_strategy import RSIStrategy

strategy = RSIStrategy({
    'period': 14,
    'oversold': 30,
    'overbought': 70
})

engine = BacktestEngine(
    data_provider=data_provider,
    strategy=strategy,
    initial_cash=1000000,
    commission_rate=0.0003,
    risk_config=risk_config
)

results = engine.run(symbols, start, end)
engine.print_results()

## 下一步

- 编辑 `config/config.yaml` 配置你的API密钥
- 使用命令行运行: `python scripts/run_backtest.py --strategy MovingAverageCrossover --symbols 000001.SH 600000.SH`
- 进行参数优化: `python scripts/optimize_parameters.py`
- 创建你自己的策略，请参考: `docs/strategies/development.md`